# Lab 06B: Advanced LLM-as-Judge Rubric Calibration

**Duration:** 30-45 minutes. Based on [Rubric evaluators - Microsoft Foundry](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/rubric-evaluators).

Define what a good Contoso support answer means, score deliberately good and flawed examples, and inspect the judge's explanation for each dimension. This notebook uses a native Foundry evaluator with `definition.type = 'rubric'`, not just a chat-completion prompt asking for a score.

Complete [Lab 04: basic evaluation](../lab-04-portal-sdk-custom/lab-04-portal-sdk-custom.ipynb), [Lab 05: RAG evaluation](../lab-05-rag-evaluation/lab-05-rag-evaluation.ipynb), and [Lab 06: tool-trajectory evaluation](lab-06-agent-evaluation.ipynb) first. This optional Lab 06B adds the **weighted rubric** workflow from the linked article after exploring answer, retrieval, and action failure modes. Your existing Professional Tone (Prompt) and Response Quality Checker (Code) entries are not overwritten.

The default examples are hand-authored calibration answers, not Lab 06 tool traces. Optional agent-based rubric generation reuses the reference agent pinned in [Lab 03B](../lab-03-evaluator-matrix/lab-03b-reference-agent-setup.ipynb), not Lab 06's separate tool-enabled target.

## Learning objectives
1. Author dimensions with stable IDs, descriptions, weights, and applicability.
2. Optionally generate a rubric from the **existing** Contoso agent instead.
3. Review the effective rubric before enabling LLM judging.
4. Inspect overall 0-1 scores, individual 1-5 scores, pass/fail, and reasons in the notebook and Foundry.
5. Keep errors and missing judgments visible rather than treating them as passes.

**Cloud calls are OFF by default.** Run all cells locally to review the exercise. For live work, use the project's normal permissions and a supported judge deployment, such as `gpt-4.1`. Rubric generation and scoring incur inference costs. This preview requires `azure-ai-projects>=2.8.0`; the repository's older minimum alone is not sufficient. No agent is created, modified, or redeployed.

In [ ]:
import json
import math
import os
import time
import uuid
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

repo_root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / '.env.template').is_file())
load_dotenv(repo_root / '.env')
output_dir = repo_root / 'labs' / 'lab-06-agent-evaluation' / 'rubric-results'

RUN_CLOUD = False  # Enable explicitly after reviewing the local exercise.
RUBRIC_SOURCE = 'manual'  # 'manual' or 'agent'; the latter generates from the existing agent.
PASS_THRESHOLD = 0.8  # Our workshop choice; the article's service default is 0.5.
project_endpoint = os.environ.get('AI_FOUNDRY_PROJECT_ENDPOINT')
tenant_id = os.environ.get('TENANT_ID')
judge_deployment = os.environ.get('AZURE_AI_MODEL_DEPLOYMENT_NAME', 'gpt-4.1')
generation_model = 'gpt-4.1'  # Supported model name for generation, not necessarily a deployment alias.
agent_name = os.environ.get('AI_FOUNDRY_AGENT_NAME', 'contoso-support-agent-eval')
agent_version = os.environ.get('AI_FOUNDRY_AGENT_VERSION')
if RUBRIC_SOURCE not in {'manual', 'agent'}:
    raise ValueError("RUBRIC_SOURCE must be 'manual' or 'agent'.")
evaluator_name = f'contoso_support_quality_{RUBRIC_SOURCE}_rubric'
print(f'Cloud enabled: {RUN_CLOUD}; rubric source: {RUBRIC_SOURCE}; judge: {judge_deployment}')
print('Local-only mode makes no Azure calls and produces no fabricated judge scores.')

## 1. Define and edit a Contoso rubric

Like the article's restaurant example, each dimension measures a separate behavior. Here, policy accuracy is the most outcome-decisive dimension. One weight is 9; the others are 1-6, following the generation heuristic (manual edits are not constrained by that heuristic).

- `id` is a stable identity. Preserve it when refining the same dimension in a future version.
- `weight` expresses relative importance; it is not a score.
- `always_applicable: true` requires a score for every response. Conditional escalation can be marked not applicable on routine questions.
- Applicable dimensions are judged on a **1-5 scale**. The service supplies a weighted, normalized **0-1 overall score**. Non-applicable dimensions do not contribute.

Treat the service's returned overall score and verdict as authoritative; do not borrow Lab 4's code-evaluator normalization formula or manufacture a verdict from rounded scores. A rubric is not a substitute for dedicated safety/groundedness checks. Add those separately when moving beyond this calibration exercise.

In [ ]:
dimensions = [
    {'id': 'policy_accuracy', 'weight': 9, 'always_applicable': False,
     'description': 'Check factual claims against the policy reference supplied in the query. 5: all numbers, conditions, and procedures are supported. 3: partly correct but incomplete or imprecise. 1: contradicts the reference or invents material policy details. Do not reward confident unsupported claims.'},
    {'id': 'question_resolution', 'weight': 6, 'always_applicable': False,
     'description': 'Answer the employee question with the relevant conditions and actionable next steps. 5: resolves the question without unnecessary clarification. 3: partially answers it. 1: misses the intent or provides unusable advice.'},
    {'id': 'action_honesty', 'weight': 5, 'always_applicable': False,
     'description': 'Do not claim an expense, leave request, ticket, or other action was executed without evidence. These examples are text-only and have no tool execution. 5: accurately describes procedures and limits. 1: falsely claims to have executed or approved an action.'},
    {'id': 'professional_tone', 'weight': 3, 'always_applicable': False,
     'description': 'Use respectful, professional, empathetic, concise workplace communication. 5: clear, helpful and actionable. 3: understandable but robotic or unnecessarily verbose. 1: dismissive, rude, or inappropriate.'},
    {'id': 'appropriate_escalation', 'weight': 4, 'always_applicable': False,
     'description': 'Applicable when the question involves a sensitive employment matter or needs a human decision. 5: directs the employee to the appropriate human team without claiming authority or requesting unnecessary sensitive details. 1: fails to refer a sensitive matter or falsely claims to resolve it. Routine policy questions may be not applicable.'},
    {'id': 'general_quality', 'weight': 5, 'always_applicable': True,
     'description': 'Other important quality factors not already covered by the listed criteria.'},
]

def validate_dimensions(values, threshold):
    if not values or not 0 <= threshold <= 1:
        raise ValueError('Provide dimensions and a pass threshold in [0, 1].')
    ids = [entry['id'] for entry in values]
    if len(ids) != len(set(ids)):
        raise ValueError('Dimension IDs must be unique.')
    for entry in values:
        if not entry['id'].strip() or not entry['description'].strip():
            raise ValueError('Dimension IDs and descriptions cannot be empty.')
        if type(entry['weight']) is not int or not 1 <= entry['weight'] <= 10:
            raise ValueError('Dimension weights must be integers in [1, 10].')
        if type(entry.get('always_applicable', False)) is not bool:
            raise ValueError('always_applicable must be boolean.')

validate_dimensions(dimensions, PASS_THRESHOLD)
display(pd.DataFrame(dimensions))

## 2. Inspect contrasting responses before asking the judge

These are **hand-authored calibration examples**, not new outputs from the existing agent. The first three use policy excerpts from the golden dataset. The final case uses an explicitly labeled workshop rule to demonstrate conditional escalation.

The policy reference is deliberately included in the judge's `query`, so the native rubric receives the evidence through the documented query/response mapping. Reference answers (`ground_truth`) are not substituted for an evaluated response. `human_expectation` is a discussion aid and is **not sent to the judge**.

Predict which dimensions should distinguish an accurate answer from a confident hallucination. The LLM's precise scores are not guaranteed and should be calibrated against human review.

In [ ]:
dataset_path = repo_root / 'datasets' / 'golden' / 'contoso-golden-v1.jsonl'
golden = [json.loads(line) for line in dataset_path.read_text(encoding='utf-8').splitlines() if line.strip()]
leave, expenses = golden[0], golden[1]
examples = [
    {'case_id': 'accurate_leave', 'query': leave['query'], 'context': leave['context'],
     'response': 'Under HR Policy 3.1, full-time employees receive 20 working days of annual leave per year, accruing at 1.67 days per month. Submit requests for more than three consecutive days at least two weeks in advance.',
     'human_expectation': 'Stronger policy accuracy and resolution; escalation may be not applicable.'},
    {'case_id': 'invented_leave_and_action', 'query': leave['query'], 'context': leave['context'],
     'response': 'You get 35 days of annual leave immediately, with no advance notice needed. I have approved your leave for you.',
     'human_expectation': 'Weaker accuracy and action honesty despite fluent wording.'},
    {'case_id': 'dismissive_expenses', 'query': expenses['query'], 'context': expenses['context'],
     'response': 'Just ask someone in Finance. I am not going to explain the process.',
     'human_expectation': 'Weaker resolution and tone; does not explain the provided policy.'},
    {'case_id': 'sensitive_hr_referral', 'query': 'How should I report workplace harassment?',
     'context': 'Workshop rule for this example: direct sensitive employment reports to HR through the confidential reporting process. Do not investigate the allegation or request unnecessary personal details.',
     'response': 'I am sorry you are dealing with this. Please contact HR through the confidential reporting process. You do not need to share private details here; HR can explain the next steps and available support.',
     'human_expectation': 'Appropriate escalation should apply and the referral should be supported.'},
]
judge_items = [{'item': {
    'case_id': example['case_id'],
    'query': f"Policy reference:\n{example['context']}\n\nEmployee question:\n{example['query']}",
    'response': example['response'],
}} for example in examples]
display(pd.DataFrame(examples)[['case_id', 'query', 'response', 'human_expectation']])
print(f'{len(judge_items)} calibration responses prepared; no judge has run yet.')

## 3. Register or generate a native rubric in your existing project

Set `RUN_CLOUD=True` in the configuration cell and rerun the cells above, then this section. Authenticate with Azure CLI using your normal project tenant before running. If the SDK is too old, install `azure-ai-projects>=2.8.0` in the notebook kernel environment and restart it.

- **manual:** register the dimensions above. On rerun, reuse the latest version only if the rubric matches; incompatible definitions raise an error, not an overwrite. Choose a new evaluator name for a deliberate revision.
- **agent:** reuse the configured existing agent/version as generation context, following the article's recommended generation workflow. A supplemental prompt explains that these calibration cases contain policy evidence but no tool trajectories. Generation automatically persists a new rubric evaluator. If one already exists under this name, reuse it rather than regenerating on every notebook run.

The agent's instructions and tools remain unchanged. The generated rubric can differ from the authored example: inspect the returned dimensions and threshold before scoring. For agent generation, the source is the agent; the **evaluation target below is the recorded calibration dataset**, not a fresh invocation of the agent.

In [ ]:
from azure.core.exceptions import ResourceNotFoundError

def resolve_rubric(catalog, name, source, authored_dimensions, threshold, generation_job=None, operation_id=None):
    try:
        versions = list(catalog.list_versions(name=name))
    except ResourceNotFoundError:
        versions = []
    if versions:
        evaluator = max(versions, key=lambda entry: int(entry.version))
        definition = evaluator.definition.as_dict()
        if definition.get('type') != 'rubric':
            raise ValueError(f'{name} already exists but is not a native rubric. Use a separate name.')
        if source == 'manual':
            actual = {entry['id']: (entry['description'], entry['weight'], entry.get('always_applicable', False)) for entry in definition['dimensions']}
            expected = {entry['id']: (entry['description'], entry['weight'], entry.get('always_applicable', False)) for entry in authored_dimensions}
            if actual != expected or definition.get('pass_threshold', 0.5) != threshold:
                raise ValueError(f'{name} has a different rubric/threshold. Review its version or choose a new name; nothing was overwritten.')
        print(f'Reusing native rubric {name}, version {evaluator.version}.')
        return evaluator
    if source == 'manual':
        return catalog.create_version(name=name, evaluator_version={
            'name': name, 'display_name': 'Contoso Support Quality Rubric (Manual)',
            'evaluator_type': 'custom', 'categories': ['quality'],
            'definition': {'type': 'rubric', 'dimensions': authored_dimensions, 'pass_threshold': threshold},
        })
    if source != 'agent' or generation_job is None or not operation_id:
        raise ValueError('Agent generation needs a job and an idempotent operation ID.')
    print(f'Generating rubric; operation ID: {operation_id}. If polling times out, retain this ID and reattach rather than start another generation.')
    poller = catalog.begin_create_generation_job(job=generation_job, operation_id=operation_id, polling_interval=10)
    return poller.result(timeout=600)

rubric_evaluator = None
effective_dimensions = dimensions
project_client = None
openai_client = None
agent_reference = None
if RUN_CLOUD:
    if tuple(int(part) for part in version('azure-ai-projects').split('.')[:2]) < (2, 8):
        raise RuntimeError('This notebook requires azure-ai-projects>=2.8.0. Upgrade the kernel environment and restart it.')
    if not project_endpoint:
        raise ValueError('AI_FOUNDRY_PROJECT_ENDPOINT is required for cloud mode.')
    from azure.identity import AzureCliCredential
    from azure.ai.projects import AIProjectClient
    preview_headers = {'Foundry-Features': 'Evaluations=V1Preview'}
    project_client = AIProjectClient(endpoint=project_endpoint, credential=AzureCliCredential(tenant_id=tenant_id), headers=preview_headers)
    openai_client = project_client.get_openai_client(default_headers=preview_headers)
    output_dir.mkdir(parents=True, exist_ok=True)
    job = None
    operation_id = None
    if RUBRIC_SOURCE == 'agent':
        if agent_version:
            existing_agent = project_client.agents.get_version(agent_name=agent_name, agent_version=agent_version)
        else:
            existing_agent = project_client.agents.get(agent_name=agent_name).versions.latest
        agent_reference = {'name': existing_agent.name, 'version': existing_agent.version}
        operation_path = output_dir / 'generation-operation.json'
        generation_request = {'project_endpoint': project_endpoint, 'evaluator_name': evaluator_name, 'agent': agent_reference, 'model': generation_model}
        if operation_path.exists():
            saved_request = json.loads(operation_path.read_text(encoding='utf-8'))
            if saved_request['request'] != generation_request:
                raise ValueError('A saved generation operation has different inputs. Review generation-operation.json before starting a different job.')
            operation_id = saved_request['operation_id']
        else:
            operation_id = f'contoso-rubric-{uuid.uuid4().hex}'
            operation_path.write_text(json.dumps({'request': generation_request, 'operation_id': operation_id}, indent=2), encoding='utf-8')
        job = {'model': generation_model, 'evaluator_name': evaluator_name,
               'evaluator_display_name': 'Contoso Support Quality Rubric (Generated)',
               'sources': [
                   {'type': 'agent', 'agent_name': existing_agent.name, 'agent_version': existing_agent.version},
                   {'type': 'prompt', 'prompt': 'Generate a quality rubric for Contoso HR, IT and Finance support. Judge factual claims against policy references supplied in the query, honest limits on actions, professional tone, and appropriate human escalation. These single-turn calibration responses contain no tool execution evidence; do not infer successful tool calls.'},
               ]}
    rubric_evaluator = resolve_rubric(project_client.evaluators, evaluator_name, RUBRIC_SOURCE, dimensions, PASS_THRESHOLD, job, operation_id)
    if rubric_evaluator.definition.type != 'rubric':
        raise RuntimeError('The service did not return a native rubric evaluator.')
    effective_dimensions = rubric_evaluator.definition.as_dict()['dimensions']
    (output_dir / 'rubric-definition.json').write_text(json.dumps(rubric_evaluator.as_dict(), indent=2, default=str), encoding='utf-8')
    print(f'Catalog evaluator: {rubric_evaluator.name}, version {rubric_evaluator.version}')
    print(f'Effective pass threshold: {rubric_evaluator.definition.pass_threshold}')
else:
    print('NOT RUN: cloud registration/generation. Set RUN_CLOUD=True to enable it.')
display(pd.DataFrame(effective_dimensions))

## 4. Human review checkpoint

Review the **effective** dimensions above, especially when generated. Do they measure the intended behavior? Is policy accuracy weighted appropriately? Are any dimensions impossible to judge from these single-turn examples? Is the general-quality dimension always applicable?

Only set `RUBRIC_REVIEWED=True` after reviewing the actual catalog version. Then run the scoring cell. Rerunning that cell submits another paid evaluation, so use the recorded run IDs to inspect an existing run instead of resubmitting after a timeout.

For manual revisions, preserve dimension IDs, choose an explicitly new evaluator name, and review the changed criteria. Never compare scores across changed rubric definitions as if they were the same metric.

In [ ]:
RUBRIC_REVIEWED = False
reviewed_version = (rubric_evaluator.name, rubric_evaluator.version) if RUBRIC_REVIEWED and rubric_evaluator is not None else None
print(f'Review approved for catalog version: {reviewed_version}')

In [ ]:
def wait_for_rubric_run(client, eval_id, run, timeout_seconds=1200, poll_seconds=10):
    deadline = time.monotonic() + timeout_seconds
    while run.status not in {'completed', 'failed', 'canceled', 'cancelled'}:
        if time.monotonic() >= deadline:
            raise TimeoutError(f'Run {run.id} is still {run.status}. It was not canceled. Retrieve it using eval ID {eval_id}; do not submit another run.')
        time.sleep(poll_seconds)
        run = client.evals.runs.retrieve(eval_id=eval_id, run_id=run.id)
        print(f'Run status: {run.status}')
    if run.status != 'completed':
        raise RuntimeError(f'Run {run.id}: {run.status}; error={getattr(run, "error", None)}')
    return run

eval_run = None
eval_object = None
criterion_name = 'Contoso Support Rubric'
if RUN_CLOUD and RUBRIC_REVIEWED:
    if rubric_evaluator is None or reviewed_version != (rubric_evaluator.name, rubric_evaluator.version):
        raise ValueError('Review the current catalog evaluator/version before scoring.')
    from openai.types.eval_create_params import DataSourceConfigCustom
    testing_criteria = [{
        'type': 'azure_ai_evaluator', 'name': criterion_name,
        'evaluator_name': rubric_evaluator.name, 'evaluator_version': rubric_evaluator.version,
        'initialization_parameters': {'deployment_name': judge_deployment},
        'data_mapping': {'query': '{{item.query}}', 'response': '{{item.response}}'},
    }]
    eval_object = openai_client.evals.create(
        name='Contoso LLM-as-Judge - Rubric Calibration',
        data_source_config=DataSourceConfigCustom(type='custom', include_sample_schema=True, item_schema={
            'type': 'object', 'properties': {key: {'type': 'string'} for key in ('case_id', 'query', 'response')},
            'required': ['case_id', 'query', 'response'],
        }),
        testing_criteria=testing_criteria,
    )
    eval_run = openai_client.evals.runs.create(
        eval_id=eval_object.id, name=f'Contoso rubric calibration - v{rubric_evaluator.version}',
        data_source={'type': 'jsonl', 'source': {'type': 'file_content', 'content': judge_items}},
    )
    run_reference = {'project_endpoint': project_endpoint, 'eval_id': eval_object.id, 'run_id': eval_run.id,
                     'evaluator_name': rubric_evaluator.name, 'evaluator_version': rubric_evaluator.version,
                     'judge_deployment': judge_deployment, 'rubric_source': RUBRIC_SOURCE, 'generation_agent': agent_reference,
                     'input_kind': 'hand_authored_calibration_responses'}
    (output_dir / 'run-reference.json').write_text(json.dumps(run_reference, indent=2), encoding='utf-8')
    print(f'Evaluation: {eval_object.id}; run: {eval_run.id}')
    eval_run = wait_for_rubric_run(openai_client, eval_object.id, eval_run)
    print(f'Foundry report: {eval_run.report_url}')
else:
    print('NOT RUN: cloud judging requires RUN_CLOUD=True and review approval for the current rubric version.')

## 5. Inspect overall results AND dimension-level reasons

The article's output includes `score`, `label`, `passed`, `reason`, and `properties.dimension_scores`. Each dimension includes its ID, applicability, weight, score, and reason. A non-applicable dimension is not a zero and is not a pass.

The next cell retains raw results and exposes missing, duplicate, skipped, or errored judgments. `complete_for_review` means that the expected judgments and explanations were returned, **not** that this four-case exercise is a statistically valid release baseline. No existing Lab 4/Lab 7 artifacts are overwritten.

In [ ]:
def rubric_result_tables(items, expected_dimensions, expected_cases, name):
    dimension_lookup = {entry['id']: entry for entry in expected_dimensions}
    overall_rows, dimension_rows, issues = [], [], []
    case_ids = []
    for item in items:
        source = item['datasource_item']
        source = source.get('item', source)
        case_id = source['case_id']
        case_ids.append(case_id)
        results = [result for result in item.get('results', []) if result['name'] == name]
        if len(results) != 1:
            issues.append(f'{case_id}: expected one rubric result; got {len(results)}. All raw records are retained.')
        sample_error = (item.get('sample') or {}).get('error')
        if sample_error:
            issues.append(f'{case_id}: sample error: {sample_error}')
        for result in results:
            error = result.get('error') or (result.get('sample') or {}).get('error')
            status = result.get('status', 'completed')
            score = result.get('score')
            if error or status in {'error', 'errored', 'skipped'}:
                issues.append(f'{case_id}: {status}: {error or result.get("reason")}')
            elif type(score) not in (float, int) or not math.isfinite(score) or not 0 <= score <= 1 or type(result.get('passed')) is not bool:
                issues.append(f'{case_id}: missing or invalid overall score/verdict.')
            if not result.get('reason'):
                issues.append(f'{case_id}: missing overall explanation.')
            overall_rows.append({'case_id': case_id, 'response': source['response'], 'status': status,
                                 'score': score, 'label': result.get('label'), 'passed': result.get('passed'),
                                 'threshold': result.get('threshold'), 'reason': result.get('reason'), 'error': error})
            scored_dimensions = (result.get('properties') or {}).get('dimension_scores', [])
            ids = [entry['id'] for entry in scored_dimensions]
            if len(ids) != len(set(ids)) or set(ids) != set(dimension_lookup):
                issues.append(f'{case_id}: missing, duplicate or unexpected dimension judgments.')
            for dimension in scored_dimensions:
                definition = dimension_lookup.get(dimension['id'])
                applicable = dimension.get('applicable')
                value = dimension.get('score')
                if type(applicable) is not bool:
                    issues.append(f'{case_id}/{dimension["id"]}: missing applicability.')
                if applicable is True and (type(value) not in (int, float) or not math.isfinite(value) or not 1 <= value <= 5):
                    issues.append(f'{case_id}/{dimension["id"]}: invalid applicable dimension score.')
                if definition and (dimension.get('weight') != definition['weight'] or (definition.get('always_applicable') and applicable is not True)):
                    issues.append(f'{case_id}/{dimension["id"]}: weight or always-applicable contract mismatch.')
                if not dimension.get('reason'):
                    issues.append(f'{case_id}/{dimension["id"]}: missing dimension explanation.')
                dimension_rows.append({'case_id': case_id, **dimension})
    if sorted(case_ids) != sorted(expected_cases):
        issues.append('Missing, duplicate or unexpected calibration cases.')
    return pd.DataFrame(overall_rows), pd.DataFrame(dimension_rows), issues

if eval_run is not None:
    output_items = [item.model_dump(mode='json') for item in openai_client.evals.runs.output_items.list(eval_id=eval_object.id, run_id=eval_run.id)]
    payload = {**run_reference, 'timestamp': datetime.now(timezone.utc).isoformat(),
               'report_url': eval_run.report_url, 'run': eval_run.model_dump(mode='json'),
               'rubric': rubric_evaluator.as_dict(), 'input_items': judge_items, 'output_items': output_items}
    results_path = output_dir / 'rubric-evaluation-results.json'
    results_path.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
    overall_results, dimension_results, review_issues = rubric_result_tables(output_items, effective_dimensions, [entry['case_id'] for entry in examples], criterion_name)
    for criterion in eval_run.per_testing_criteria_results or []:
        if getattr(criterion, 'errored', 0) or getattr(criterion, 'skipped', 0):
            review_issues.append(f'{criterion.testing_criteria}: run summary contains errored/skipped items.')
    if not eval_run.report_url:
        review_issues.append('The service did not return a portal report URL.')
    payload.update(complete_for_review=not review_issues, review_issues=review_issues, valid_for_release_gate=False)
    results_path.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
    display(overall_results)
    display(dimension_results)
    for issue in review_issues:
        print(f'WARNING: {issue}')
    print(f'Complete for human review: {not review_issues}; valid for release gate: False (calibration only).')
    print(f'Saved actual cloud results: {results_path}')
else:
    print('NOT RUN: no cloud evaluation exists. The example responses above are not judge results.')

## 6. Find the evaluator and results in Foundry

1. Open your existing project, then **Evaluations > Evaluator catalog**. Find `contoso_support_quality_manual_rubric` or `contoso_support_quality_agent_rubric`. Inspect its native rubric definition, version, dimensions, weights, and threshold. This is separate from the Prompt/Code entries shown in the screenshot.
2. Open the service-provided **Foundry report URL** printed above. If navigation differs, find the evaluation/run ID on the project-level **Evaluations** page. This is automated evaluation, not Human Evaluation.
3. Inspect a high-quality answer and a flawed answer. Expand rubric details where the portal exposes them; the dimension table and saved `properties.dimension_scores` remain available in the notebook even if a portal layout does not display all fields.
4. Compare your initial human expectations with the LLM reasons. A fluent hallucination should not receive credit for policy accuracy merely because it sounds confident. Do not assume exact scores or rankings before observing them.

## Learner experiments

- Explain why routine annual-leave questions may have escalation marked not applicable, but the HR referral example should not.
- Propose a more precise description for a dimension on which you disagree with the judge. Preserve its ID, save under an explicitly new evaluator name, and compare reasons as well as scores. A changed rubric is a changed metric, not an agent regression.
- Try `RUBRIC_SOURCE='agent'` to generate a rubric from the same existing agent. Review it before running; generated dimensions, weights, and thresholds need not match the manual example.
- For a later real-agent evaluation, use actual recorded agent responses or an agent-target cloud run. Do not relabel these hand-authored examples as live agent results. Retain blocked/error/skipped cases and inspect them rather than bypassing content filters.
- Pair the reviewed rubric with dedicated built-in groundedness and safety evaluators before production use. Calibrate on a larger, representative held-out dataset before introducing any release gate.

Cloud resources are deliberately kept for the portal demonstration. No cleanup cell automatically deletes an evaluator or run that other learners may be using. Close the clients when finished; delete only resources you intentionally created and no longer need.

## References
- [Rubric evaluator concepts, weights, applicability, and example output](https://learn.microsoft.com/azure/foundry/concepts/evaluation-evaluators/rubric-evaluators)
- [Official SDK manual-authoring sample](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_rubric_evaluator_manual.py)
- [Official SDK generation sample](https://github.com/Azure/azure-sdk-for-python/blob/main/sdk/ai/azure-ai-projects/samples/evaluations/sample_rubric_evaluator_generation_basic.py)
- [Lab 4 learning paths and portal model exercise](../lab-04-portal-sdk-custom/README.md)
- [Next: Lab 07 CI/CD evaluation gate](../lab-07-cicd/lab-07-cicd-evaluation-gate.ipynb)

In [ ]:
if openai_client is not None:
    openai_client.close()
if project_client is not None:
    project_client.close()
print('Clients closed; no cloud evaluators, agents, or runs were deleted.')